In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import os
import re
import datetime
import urllib3
import requests
import pandas as pd
from bs4 import BeautifulSoup

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
#------------------------------------------------ Begin_fileName ----------------------------------------
regulatorName = 'US FED'

print(f"Running {regulatorName} Web Scraping Tool v.1.3")

now = datetime.datetime.now()
processdate = now.strftime('%Y-%m-%d')
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

# ------ At first we will define the workspace path -----
try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__))  ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd()  ## notebook environment

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder')  # if files are downloaded during the process
if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running US FED Web Scraping Tool v.1.3


In [3]:
#------------------------------------------------ Begin_session ----------------------------------------
# v1.3 : Selenium/ChromeDriver dropped - both FED sources are static server-rendered HTML.
# verify=False is required behind the corporate TLS proxy.
HEADERS = {
    'User-Agent': ('Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
                   '(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36'),
    'Accept-Language': 'en-US,en;q=0.9',
}

session = requests.Session()
session.headers.update(HEADERS)
session.verify = False


def get_soup(url, timeout=60):
    resp = session.get(url, timeout=timeout)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, 'html.parser')

In [4]:
regdict = {
    'US FED 1': 'https://www.federalreserve.gov/releases/lbr/current/default.htm',
    'US FED 3': 'https://www.federalreserve.gov/releases/iba/',
}

sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
          'Phone - Mother company': []}

# v1.3 : added ARMENIA / DOMINICAN REPUBLIC / 'UNITED KINGDOM  (OTHER)' (double space, as published)
#        and corrected UNITED KINGDOM -> GB (was 'UK', not a valid ISO-3166 alpha-2 code).
ISO = {'ARGENTINA': 'AR', 'ARMENIA': 'AM', 'AUSTRALIA': 'AU', 'AUSTRIA': 'AT', 'BAHRAIN': 'BH', 'BELGIUM': 'BE', 'BRAZIL': 'BR',
       'CANADA': 'CA', 'CHILE': 'CL', 'CHINA, PEOPLES REPUBLIC OF': 'CN', 'COLOMBIA': 'CO',
       'CURACAO, BONAIRE, SABA, ST. MARTIN & ST.': 'CW', 'DOMINICAN REPUBLIC': 'DO', 'ECUADOR': 'EC', 'EGYPT': 'EG',
       'FINLAND': 'FI', 'FRANCE': 'FR',
       'GERMANY': 'DE', 'HONDURAS': 'HN', 'HONG KONG': 'HK', 'INDIA': 'IN', 'INDONESIA': 'ID', 'IRELAND': 'IE',
       'ISRAEL': 'IL', 'ITALY': 'IT', 'JAMAICA': 'JM', 'JAPAN': 'JP', 'JORDAN': 'JO', 'KOREA, SOUTH': 'KR', 'KUWAIT': 'KW',
       'LUXEMBOURG': 'LU', 'MALAYSIA': 'MY', 'MEXICO': 'MX', 'MOROCCO (OTHER)': 'MA', 'NETHERLANDS': 'NL', 'NIGERIA': 'NG',
       'NORWAY': 'NO', 'PAKISTAN': 'PK', 'PANAMA': 'PA', 'PERU': 'PE', 'PHILIPPINES': 'PH', 'PORTUGAL': 'PT',
       'SAUDI ARABIA': 'SA', 'SINGAPORE': 'SG', 'SOUTH AFRICA': 'ZA', 'SPAIN': 'ES', 'SWEDEN': 'SE', 'SWITZERLAND': 'CH',
       'TAIWAN': 'TW', 'THAILAND': 'TH', 'TURKEY': 'TR', 'UKRAINE': 'UA', 'UNITED ARAB EMIRATES': 'AE',
       'UNITED KINGDOM': 'GB', 'UNITED KINGDOM  (OTHER)': 'GB', 'UNITED STATES': 'US', 'URUGUAY': 'UY', 'VIETNAM': 'VN'}

Typology = {
    "US FED 1": "Large Commercial Banks",
    "US FED 3": "Foreign Banking Organization as a BHC",
}

# ListLabel : 1 = bank lists, 2 = insurance, 3 = bank & insurance, 4 = everything else.
ListLabel = {
    "US FED 1": 1,
    "US FED 3": 1,
}

empty_ = ''

In [5]:
#------------------------------------------------ Begin_Fonction ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key] = sqldict[key] + empty
    return sqldict


def split_city(location):
    """'COLUMBUS, OH' -> 'COLUMBUS'. Returns '' when the pattern does not hold."""
    if not location:
        return ''
    return location.split(',')[0].strip()


def clean(text):
    """Collapse internal newlines / runs of whitespace that the FED tables carry."""
    return ' '.join(text.split())


# v1.3 : IBA names are published as "ENTITY NAME (CODE - DESCRIPTION)", e.g.
#        "BANCO NACION ARGENTINA NY BR (USB - UNINSURED STATE BRANCH)".
#        The suffix is an office-type annotation, not part of the name - split it out
#        so Name holds the entity and Typology/CoType hold the office type.
IBA_SUFFIX = re.compile(r'^(.*?)\s*\(([A-Z]{2,4})\s*-\s*(.+?)\)\s*$')


def split_iba_name(raw):
    """'X NY BR (USB - UNINSURED STATE BRANCH)' -> ('X NY BR', 'USB', 'UNINSURED STATE BRANCH')"""
    m = IBA_SUFFIX.match(raw)
    if m:
        return m.group(1).strip(), m.group(2).strip(), m.group(3).strip()
    return raw, '', ''

In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------
for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} | {reg} ")

    if reg == 'US FED 1':
        soup = get_soup(regdict[reg])

        # v1.3 : the LBR table lost its cellspacing="0" attribute and is now class="pubtables".
        table = soup.find("table", class_="pubtables")
        if table is None:
            raise Exception(f'[ERROR] : - LBR table not found - selector drift again | {reg}')

        # v1.3 : data-as-of date drives ListValidityDate (e.g. "March 31, 2026").
        validity = ''
        m = re.search(r'([A-Z][a-z]+\s+\d{1,2},\s+\d{4})', soup.get_text())
        if m:
            try:
                validity = datetime.datetime.strptime(m.group(1), '%B %d, %Y').strftime('%Y-%m-%d')
            except ValueError:
                validity = ''

        body = table.find('tbody') or table
        trs = body.find_all('tr')
        print(f'[INFO] : - Data Scrapping = {len(trs)} | {reg}')

        for tr in trs:
            th = tr.find('th')
            tds = tr.find_all('td')

            # v1.3 : bank name moved into <th scope="row">; data rows now carry exactly 11 <td>.
            if th is None or len(tds) < 11:
                continue

            # "JPMORGAN CHASE BK NA / JPMORGAN CHASE & CO" -> entity / holding company
            fullname = clean(th.get_text())
            if ' / ' in fullname:
                name, mother = [p.strip() for p in fullname.split(' / ', 1)]
            else:
                name, mother = fullname, ''

            location = clean(tds[2].get_text())

            sqldict['Name'].append(name)
            sqldict['Name - Mother Company'].append(mother)
            sqldict['InternalID_1'].append(clean(tds[1].get_text()))
            sqldict['InternalID_1_type'].append('Bank ID')
            sqldict['Address_1'].append(location)
            sqldict['City'].append(split_city(location))
            sqldict['Typology'].append(clean(tds[3].get_text()))
            sqldict['ListProcessDate'].append(processdate)
            sqldict['ListValidityDate'].append(validity)
            sqldict['ListLanguage'].append('EN')
            sqldict['ListLabel'].append(ListLabel[reg])
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append('US')
            sqldict['RegCode'].append('FED')
            sqldict['Cntry'].append('US')
            sqldict['ListCode'].append(reg[-1])
            sqldict['ListName'].append(Typology[reg])

        sqldict = bourange_same_length_array(sqldict)

    elif reg == 'US FED 3':
        # v1.3 : the two Selenium clicks (brittle absolute XPath + PARTIAL_LINK_TEXT) are replaced
        #        by direct discovery of the most recent YYYYMM release directory.
        idx = get_soup(regdict[reg])
        releases = sorted({m.group(1)
                           for a in idx.find_all('a', href=True)
                           for m in [re.search(r'(\d{6})/default\.htm', a['href'])]
                           if m}, reverse=True)
        if not releases:
            raise Exception(f'[ERROR] : - no YYYYMM release directory found on {regdict[reg]} | {reg}')

        latest = releases[0]
        print(f'[INFO] : - most recent release = {latest} | {reg}')
        validity = f'{latest[:4]}-{latest[4:]}-01'

        soup = get_soup(f'https://www.federalreserve.gov/releases/iba/{latest}/bycntry.htm')

        ps = soup.find_all("p", {"class": "ST4"})
        tables = soup.find_all("table", {"border": "1"})
        if len(ps) != len(tables):
            raise Exception(f'[ERROR] : - {len(ps)} country headings vs {len(tables)} tables - misaligned | {reg}')

        for i in range(len(ps)):
            country = ps[i].text.replace('COUNTRY:', '').strip()
            if country not in ISO:
                print(f'[WARN] : - country missing from ISO map -> Cntry blank : "{country}"')

            trs = tables[i].find_all('tr')
            print(f'[INFO] : - Data Scrapping in table {i+1}/{len(ps)} = {len(trs)} | {reg}')

            for j in range(1, len(trs)):
                tds = trs[j].find_all('td')
                if len(tds) <= 2:
                    continue

                location = clean(tds[2].get_text())

                name, cotype, officetype = split_iba_name(clean(tds[1].get_text()))
                mother, _, _ = split_iba_name(clean(tds[0].get_text()))

                sqldict['Name'].append(name)
                sqldict['CoType'].append(cotype)
                sqldict['Typology'].append(officetype)
                sqldict['Name - Mother Company'].append(mother)
                sqldict['Address_1'].append(location)
                sqldict['City'].append(split_city(location))
                sqldict['ListProcessDate'].append(processdate)
                sqldict['ListValidityDate'].append(validity)
                sqldict['ListLanguage'].append('EN')
                sqldict['ListLabel'].append(ListLabel[reg])
                sqldict['RegulationType'].append('Regulated')
                sqldict['RegCtry'].append('US')
                sqldict['RegCode'].append('FED')
                sqldict['Cntry'].append(ISO.get(country, ''))
                sqldict['Cntry - Mother company'].append(ISO.get(country, ''))
                sqldict['ListCode'].append(reg[-1])
                sqldict['ListName'].append(Typology[reg])

            sqldict = bourange_same_length_array(sqldict)

[INFO] : Working 1/2 | US FED 1 


[INFO] : - Data Scrapping = 3798 | US FED 1
[INFO] : Working 2/2 | US FED 3 


[INFO] : - most recent release = 202603 | US FED 3


[INFO] : - Data Scrapping in table 1/56 = 5 | US FED 3
[INFO] : - Data Scrapping in table 2/56 = 4 | US FED 3
[INFO] : - Data Scrapping in table 3/56 = 13 | US FED 3
[INFO] : - Data Scrapping in table 4/56 = 9 | US FED 3
[INFO] : - Data Scrapping in table 5/56 = 6 | US FED 3
[INFO] : - Data Scrapping in table 6/56 = 6 | US FED 3
[INFO] : - Data Scrapping in table 7/56 = 16 | US FED 3
[INFO] : - Data Scrapping in table 8/56 = 54 | US FED 3
[INFO] : - Data Scrapping in table 9/56 = 7 | US FED 3
[INFO] : - Data Scrapping in table 10/56 = 22 | US FED 3
[INFO] : - Data Scrapping in table 11/56 = 7 | US FED 3
[INFO] : - Data Scrapping in table 12/56 = 4 | US FED 3
[INFO] : - Data Scrapping in table 13/56 = 5 | US FED 3
[INFO] : - Data Scrapping in table 14/56 = 4 | US FED 3
[INFO] : - Data Scrapping in table 15/56 = 4 | US FED 3
[INFO] : - Data Scrapping in table 16/56 = 4 | US FED 3
[INFO] : - Data Scrapping in table 17/56 = 34 | US FED 3
[INFO] : - Data Scrapping in table 18/56 = 35 | US F

In [7]:
#------------------------------------------------ Begin_writer and save df to excel ----------------------------------------
os.chdir(scriptfolder)

df = pd.DataFrame(sqldict)
df = df.drop_duplicates()
df = df[df['Name'] != '']

# v1.3 : ExcelWriter/writer.save() removed - save() was dropped in pandas >= 2.0.
df.to_excel(os.path.join(scriptfolder, filename), sheet_name='SQL Ready', index=False)

print('Saved {} rows to {}'.format(len(df), os.path.join(scriptfolder, filename)))
print(df.groupby('ListCode').size())

Saved 4169 rows to /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/US FED/US FED SQL Ready 2026-07-29 10.12.42.xlsx
ListCode
1    3798
3     371
dtype: int64
